## 朴素贝叶斯：
朴素贝叶斯是基于贝叶斯定理+特征条件独立强假设的简单概率型分裂期，通过训练数据估计联合概率密度P(X,Y),最终预测标签y

- 贝叶斯原理：
$$
P(y|x)=\frac{P(x|y)P(y)}{P(x)}
$$
其中P(y)直接用古典概型来做，就是类似抛硬币方法

- P(x|y)估计：
$$
\begin{align*}
h(x)&=\argmax_{y}\ P(y|x)
    &=\argmax_{y}\ P(x|y)P(y)
    &=\argmax_{y} \sum_{\alpha=1}^d log(P(x_{\alpha}|y))+log(P(y))
\end{align*}
$$
- 简单来讲，就是比较$p(x|y_i)p(y_i)$的大小，挑选最大的那一个，因为对应的P(x)都是相同的所以我们省略掉，至于因为x也有好多个，所以要分别计算$p(x_j|y_i)$，最后得到乘积，同时可以尝试用对数形式去化简

## 正则化：防止过拟合
通过在建模的损失函数前面加上惩罚项，防止过拟合，其中正则化有两种，分别是L1正则和L2正则

In [ ]:
import numpy as np


class NaiveBayesClassifier(object):
    def __init__(self):
        self.label_prob = {}
        self.condition_prob = {}
        # 存储所有标签
        self.labels = None
        # 拉普拉斯平滑系数，这里实验不用使用的，但是普及一下
        self.alpha = 1

    def fit(self, feature, label):
        self.labels = np.unique(label)        # 去重
        total_samples = len(label)        #得到总样本数

        for lab in self.labels:
            lab_count = np.sum(label == lab)
            self.label_prob[lab] = (lab_count + self.alpha) / (total_samples + len(self.labels) * self.alpha) 
            
        for lab in self.labels:
            lab_features = feature[label == lab]
            lab_total = len(lab_features)
            self.condition_prob[lab] = {}

            for col_idx in range(feature.shape[1]):
                col_values = lab_features[:, col_idx]
                unique_vals = np.unique(feature[:, col_idx])
                self.condition_prob[lab][col_idx] = {}
                
                for val in unique_vals:
                    val_count = np.sum(col_values == val)
                    prob = (val_count + self.alpha) / (lab_total + len(unique_vals) * self.alpha)
                    self.condition_prob[lab][col_idx][val] = prob
        #********* End *********#

    def predict(self, feature):
        predictions = []
        for sample in feature: # 每一行就是一个样本
            post_probs = {}
            for lab in self.labels: #之后就是计算P(y|x),计算公式为P(y1|x)=P(y1)*P(x|y1)/p(x),但是每个类的P(x)都是相同的，只需要比较P(y1)*p(x|y1)
                prob = np.log(self.label_prob[lab]) # 变成对数形式，防止最后得到数过小，变成加法形式,但是因为没用这个拉普拉斯平滑，所以最后还是选择了乘法形式
                for col_idx, val in enumerate(sample):
                    prob+=np.log(self.condition_prob[lab][col_idx][val])
                
                post_probs[lab] = prob
            
            pred_lab = max(post_probs, key=post_probs.get)
            predictions.append(pred_lab)
        
        return np.array(predictions)
        #********* End *********#

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfTransformer


def news_predict(train_sample, train_label, test_sample):
    '''
    训练模型并进行预测，返回预测结果
    :param train_sample:原始训练集中的新闻文本,类型为ndarray
    :param train_label:训练集中新闻文本对应的主题标签,类型为ndarray
    :param test_sample:原始测试集中的新闻文本,类型为ndarray
    :return 预测结果,类型为ndarray
    '''
    #********* Begin *********#
    vec=CountVectorizer()
    X_train=vec.fit_transform(train_sample)
    X_test=vec.transform(test_sample)
    tfidf=TfidfTransformer()
    X_train=tfidf.fit_transform(X_train)
    X_test=tfidf.transform(X_test)
    clf=MultinomialNB(alpha=0.01)
    clf.fit(X_train,train_label)
    result=clf.predict(X_test)
    return result
    #********* End *********#


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

def news_predict(train_sample, train_label, test_sample):
    '''
    训练模型并进行预测，返回预测结果
    :param train_sample:原始训练集中的新闻文本,类型为ndarray
    :param train_label:训练集中新闻文本对应的主题标签,类型为ndarray
    :param test_sample:原始测试集中的新闻文本,类型为ndarray
    :return 预测结果,类型为ndarray
    '''
    #********* Begin *********#
    # 1. 用TfidfVectorizer一步完成词频+TF-IDF，同时过滤停用词
    vec = TfidfVectorizer(
        stop_words='english',  # 过滤英文停用词（如the, a, is）
        max_features=10000,    # 只保留出现频率最高的10000个词，减少噪声
        ngram_range=(1, 2)     # 同时考虑1-gram（单个词）和2-gram（词组，如"machine learning"）
    )
    X_train = vec.fit_transform(train_sample)
    X_test = vec.transform(test_sample)
    
    # 2. 初始化朴素贝叶斯模型，alpha可以后续调参
    clf = MultinomialNB(alpha=0.05)
    clf.fit(X_train, train_label)
    
    # 3. 预测并返回结果
    result = clf.predict(X_test)
    return result
    #********* End *********#